# Notebook 04 — Hierarchical Clustering & Perbandingan

**Tujuan:** Melakukan clustering desa/kelurahan menggunakan Hierarchical (Agglomerative) Clustering dengan Ward Linkage, lalu membandingkan hasilnya dengan K-Means untuk menentukan algoritma terbaik.

Tahapan:
1. Load data preprocessing & metrik K-Means
2. Dendrogram Ward Linkage (full & truncated)
3. Training AgglomerativeClustering
4. Evaluasi 3 metrik
5. Visualisasi PCA 2D
6. Profil klaster
7. Tabel perbandingan K-Means vs Hierarchical
8. Penentuan algoritma terbaik (voting)
9. Simpan semua hasil

In [1]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

warnings.filterwarnings('ignore')

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import FEATURE_COLS, FIGURES_DIR, RESULTS_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10})

SCALED_COLS = [f'{c}_z' for c in FEATURE_COLS]

print('Library berhasil diimport.')
print(f'RESULTS: {RESULTS_DIR.resolve()}')

Library berhasil diimport.
RESULTS: analysis\output\results


In [2]:
# ── Cell 3: Load Data ──────────────────────────────────────────────────────────
scaled_path  = RESULTS_DIR / 'dtsen_scaled.csv'
clean_path   = RESULTS_DIR / 'dtsen_clean.csv'
metrics_km_path = RESULTS_DIR / 'metrics_kmeans.json'

for p in [scaled_path, clean_path, metrics_km_path]:
    if not p.exists():
        raise FileNotFoundError(
            f'File tidak ditemukan: {p}\n'
            'Jalankan NB02_Preprocessing.ipynb dan NB03_KMeans.ipynb terlebih dahulu!'
        )

df_scaled  = pd.read_csv(scaled_path, index_col='KELURAHAN')
df_clean   = pd.read_csv(clean_path)

with open(metrics_km_path) as f:
    metrics_km = json.load(f)

X = df_scaled[SCALED_COLS].values

print(f'df_scaled shape  : {df_scaled.shape}')
print(f'df_clean shape   : {df_clean.shape}')
print(f'X shape          : {X.shape}')
print(f'\nMetrik K-Means (dari NB03):')
for k, v in metrics_km.items():
    print(f'  {k:30s}: {v}')

df_scaled shape  : (371, 10)
df_clean shape   : (371, 11)
X shape          : (371, 8)

Metrik K-Means (dari NB03):
  algorithm                     : K-Means
  k                             : 3
  silhouette_score              : 0.181573
  davies_bouldin_score          : 1.81449
  calinski_harabasz_score       : 101.595339
  inertia                       : 1912.1881


In [3]:
# ── Cell 4: Dendrogram Full ────────────────────────────────────────────────────
Z = linkage(X, method='ward')
print(f'Linkage matrix shape: {Z.shape}')
print(f'Merge count         : {len(Z)} (= n_samples - 1)')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    Z,
    ax=ax,
    no_labels=True,
    color_threshold=0,
    above_threshold_color='#4C9BE8',
)
ax.set_xlabel('Desa/Kelurahan (indeks)')
ax.set_ylabel('Ward Linkage Distance')
ax.set_title('Dendrogram Hierarki Lengkap — Ward Linkage', fontweight='bold')
plt.tight_layout()
outpath = FIGURES_DIR / '04_dendrogram_full.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Linkage matrix shape: (370, 4)
Merge count         : 370 (= n_samples - 1)


Gambar disimpan: analysis\output\figures\04_dendrogram_full.png


In [4]:
# ── Cell 5: Dendrogram Truncated (p=30 last merges) ───────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
dg = dendrogram(
    Z,
    ax=ax,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90,
    leaf_font_size=8,
    show_contracted=True,
)

# Tampilkan beberapa garis horizontal kandidat cut-off
last_merges = Z[-10:, 2]  # 10 jarak terakhir
for dist in last_merges:
    ax.axhline(y=dist, color='red', linestyle='--', linewidth=0.7, alpha=0.4)

ax.set_xlabel('Node (jumlah sampel dalam klaster)')
ax.set_ylabel('Ward Linkage Distance')
ax.set_title(
    'Dendrogram Terpotong (30 Penggabungan Terakhir) — Ward Linkage\n'
    'Garis merah = kandidat cut-off',
    fontweight='bold'
)
plt.tight_layout()
outpath = FIGURES_DIR / '04_dendrogram_truncated.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

print('\n10 jarak merge terakhir (besar = pemisahan kuat):')
for i, d in enumerate(Z[-10:, 2]):
    print(f'  Merge ke-{len(Z)-9+i}: distance = {d:.4f}')

Gambar disimpan: analysis\output\figures\04_dendrogram_truncated.png

10 jarak merge terakhir (besar = pemisahan kuat):
  Merge ke-361: distance = 10.2694
  Merge ke-362: distance = 10.9253
  Merge ke-363: distance = 11.1042
  Merge ke-364: distance = 12.0567
  Merge ke-365: distance = 12.5357
  Merge ke-366: distance = 12.9941
  Merge ke-367: distance = 13.7217
  Merge ke-368: distance = 20.3137
  Merge ke-369: distance = 22.6218
  Merge ke-370: distance = 36.9518


## Tentukan K Optimal dari Dendrogram

Perhatikan **dendrogram terpotong** di atas:
- Tarik garis horizontal di tempat yang memotong **celah terpanjang** (jarak antar merge terbesar)
- Hitung berapa cabang vertikal yang dipotong — itulah jumlah klaster
- Bandingkan dengan hasil Elbow/Silhouette dari NB03

**Set `K_OPTIMAL` pada cell berikut:**

In [5]:
# ── Cell 7: Tentukan K Optimal ────────────────────────────────────────────────
# GANTI nilai ini sesuai hasil analisis dendrogram!
K_OPTIMAL = 3  # <── SAMAKAN dengan K-Means atau sesuaikan dari dendrogram

print(f'K_OPTIMAL Hierarchical: {K_OPTIMAL}')

# Cross-check dengan K-Means
km_k = metrics_km.get('k', '?')
if K_OPTIMAL == km_k:
    print(f'✅ Konsisten dengan K-Means (k={km_k})')
else:
    print(f'⚠  Berbeda dari K-Means (k={km_k}) — akan tetap digunakan k={K_OPTIMAL} untuk Hierarchical')

K_OPTIMAL Hierarchical: 3
✅ Konsisten dengan K-Means (k=3)


In [6]:
# ── Cell 8: Training AgglomerativeClustering ───────────────────────────────────
hier_model = AgglomerativeClustering(
    n_clusters=K_OPTIMAL,
    linkage='ward',
)
hier_labels = hier_model.fit_predict(X)

df_clean = df_clean.copy()
df_clean['cluster_hier'] = hier_labels

print(f'AgglomerativeClustering(n_clusters={K_OPTIMAL}, linkage=\'ward\') — selesai')
print(f'\nDistribusi klaster:')
print(df_clean['cluster_hier'].value_counts().sort_index().to_string())

AgglomerativeClustering(n_clusters=3, linkage='ward') — selesai

Distribusi klaster:
cluster_hier
0    184
1    111
2     76


In [7]:
# ── Cell 9: Evaluasi Metrik Hierarchical ──────────────────────────────────────
sil_h  = silhouette_score(X, hier_labels)
dbi_h  = davies_bouldin_score(X, hier_labels)
chi_h  = calinski_harabasz_score(X, hier_labels)

metrics_hier = {
    'algorithm':               'Hierarchical',
    'k':                       int(K_OPTIMAL),
    'linkage':                 'ward',
    'silhouette_score':        round(float(sil_h), 6),
    'davies_bouldin_score':    round(float(dbi_h), 6),
    'calinski_harabasz_score': round(float(chi_h), 6),
}

print('='*54)
print(f'  EVALUASI HIERARCHICAL CLUSTERING (k={K_OPTIMAL})')
print('='*54)
print(f'  Silhouette Score       : {sil_h:.4f}  (↑ lebih baik, max=1)')
print(f'  Davies-Bouldin Index   : {dbi_h:.4f}  (↓ lebih baik, min=0)')
print(f'  Calinski-Harabasz Idx  : {chi_h:.4f}  (↑ lebih baik)')
print('='*54)

  EVALUASI HIERARCHICAL CLUSTERING (k=3)
  Silhouette Score       : 0.1289  (↑ lebih baik, max=1)
  Davies-Bouldin Index   : 1.9711  (↓ lebih baik, min=0)
  Calinski-Harabasz Idx  : 85.0990  (↑ lebih baik)


In [8]:
# ── Cell 10: PCA 2D Hierarchical ──────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

ev1, ev2 = pca.explained_variance_ratio_
print(f'PCA: {ev1*100:.1f}% + {ev2*100:.1f}% = {(ev1+ev2)*100:.1f}% variansi')

palette = sns.color_palette('Set2', K_OPTIMAL)

fig, ax = plt.subplots(figsize=(9, 6))
for k in range(K_OPTIMAL):
    idx = hier_labels == k
    ax.scatter(
        X_pca[idx, 0], X_pca[idx, 1],
        c=[palette[k]], label=f'Klaster {k}',
        alpha=0.7, s=45, edgecolors='white', linewidths=0.4
    )

ax.set_xlabel(f'PC1 ({ev1*100:.1f}%)')
ax.set_ylabel(f'PC2 ({ev2*100:.1f}%)')
ax.set_title(
    f'Hierarchical Clustering (k={K_OPTIMAL}, Ward) — Visualisasi PCA 2D\n'
    f'Total variansi: {(ev1+ev2)*100:.1f}%',
    fontweight='bold'
)
ax.legend(fontsize=9)
plt.tight_layout()
outpath = FIGURES_DIR / '04_pca_hierarchical.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

PCA: 39.4% + 17.4% = 56.8% variansi


Gambar disimpan: analysis\output\figures\04_pca_hierarchical.png


In [9]:
# ── Cell 11: Profil Klaster Hierarchical ──────────────────────────────────────
label_short = ['Desil1','Desil2','Desil3','Desil4','Desil5','Desil6-10','BlmPRMK','Nonaktif']

profil_h = (
    df_clean.groupby('cluster_hier')[FEATURE_COLS]
    .mean()
    .round(2)
)
profil_h.index = [f'Klaster {i}' for i in profil_h.index]

print('Profil rata-rata 8 fitur per klaster Hierarchical (%):')
print(profil_h.to_string())

fig, ax = plt.subplots(figsize=(10, max(4, K_OPTIMAL * 1.2)))
sns.heatmap(
    profil_h,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    linewidths=0.5,
    xticklabels=label_short,
    yticklabels=profil_h.index,
    cbar_kws={'label': 'Rata-rata Proporsi (%)'},
)
ax.set_title(
    f'Profil Klaster Hierarchical (k={K_OPTIMAL}, Ward)\nRata-rata Proporsi per Fitur',
    fontweight='bold'
)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
outpath = FIGURES_DIR / '04_profil_klaster_hier.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Profil rata-rata 8 fitur per klaster Hierarchical (%):
           pct_desil1  pct_desil2  pct_desil3  pct_desil4  pct_desil5  pct_desil6_10  pct_belum_prmk  pct_nonaktif
Klaster 0       18.11       16.74       11.05        8.69       11.52          30.59            3.30          8.39
Klaster 1       15.47       12.53        7.60        6.02       15.90          39.07            3.40          7.45
Klaster 2        7.42        7.22        5.86        5.29       13.65          55.90            4.67          7.72


Gambar disimpan: analysis\output\figures\04_profil_klaster_hier.png


In [10]:
# ── Cell 12: Tabel Perbandingan K-Means vs Hierarchical ───────────────────────
sil_km  = metrics_km['silhouette_score']
dbi_km  = metrics_km['davies_bouldin_score']
chi_km  = metrics_km['calinski_harabasz_score']

def winner_sil(a, b):  # ↑ lebih baik
    if a > b:   return 'K-Means ✓'
    elif b > a: return 'Hierarchical ✓'
    else:       return 'Seri'

def winner_dbi(a, b):  # ↓ lebih baik
    if a < b:   return 'K-Means ✓'
    elif b < a: return 'Hierarchical ✓'
    else:       return 'Seri'

def winner_chi(a, b):  # ↑ lebih baik
    if a > b:   return 'K-Means ✓'
    elif b > a: return 'Hierarchical ✓'
    else:       return 'Seri'

comparison_data = {
    'Metrik': [
        'Silhouette Score (↑)',
        'Davies-Bouldin Index (↓)',
        'Calinski-Harabasz Index (↑)',
    ],
    'K-Means': [
        round(sil_km, 4),
        round(dbi_km, 4),
        round(chi_km, 2),
    ],
    'Hierarchical': [
        round(sil_h, 4),
        round(dbi_h, 4),
        round(chi_h, 2),
    ],
    'Pemenang': [
        winner_sil(sil_km, sil_h),
        winner_dbi(dbi_km, dbi_h),
        winner_chi(chi_km, chi_h),
    ],
}

df_comparison = pd.DataFrame(comparison_data)

print('='*75)
print('  TABEL PERBANDINGAN: K-MEANS vs HIERARCHICAL CLUSTERING')
print('='*75)
print(df_comparison.to_string(index=False))
print('='*75)

# Simpan sebagai tabel PNG menggunakan matplotlib table
fig, ax = plt.subplots(figsize=(11, 3))
ax.axis('off')
col_widths = [0.35, 0.18, 0.18, 0.25]
tbl = ax.table(
    cellText=df_comparison.values,
    colLabels=df_comparison.columns,
    cellLoc='center',
    loc='center',
    colWidths=col_widths,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)

# Style header
for j in range(len(df_comparison.columns)):
    tbl[(0, j)].set_facecolor('#2C3E50')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Warna baris bergantian
for i in range(1, len(df_comparison) + 1):
    bg = '#EBF5FB' if i % 2 == 0 else 'white'
    for j in range(len(df_comparison.columns)):
        tbl[(i, j)].set_facecolor(bg)
        # Highlight kolom Pemenang
        val = df_comparison.iloc[i-1, 3]
        if j == 3:
            if 'K-Means' in val:
                tbl[(i, j)].set_facecolor('#FADBD8')
            elif 'Hierarchical' in val:
                tbl[(i, j)].set_facecolor('#D5F5E3')

ax.set_title(
    f'Perbandingan Metrik Evaluasi Clustering (k={K_OPTIMAL})',
    fontweight='bold', pad=15, fontsize=11
)
plt.tight_layout()
outpath = FIGURES_DIR / '04_tabel_perbandingan.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nGambar tabel disimpan: {outpath}')

  TABEL PERBANDINGAN: K-MEANS vs HIERARCHICAL CLUSTERING
                     Metrik  K-Means  Hierarchical  Pemenang
       Silhouette Score (↑)   0.1816        0.1289 K-Means ✓
   Davies-Bouldin Index (↓)   1.8145        1.9711 K-Means ✓
Calinski-Harabasz Index (↑) 101.6000       85.1000 K-Means ✓



Gambar tabel disimpan: analysis\output\figures\04_tabel_perbandingan.png


In [11]:
# ── Cell 13: Tentukan Algoritma Terbaik (Voting 3 Metrik) ────────────────────
votes_km   = 0
votes_hier = 0

# Silhouette: ↑ lebih baik
if sil_km > sil_h:   votes_km += 1
elif sil_h > sil_km: votes_hier += 1

# Davies-Bouldin: ↓ lebih baik
if dbi_km < dbi_h:   votes_km += 1
elif dbi_h < dbi_km: votes_hier += 1

# Calinski-Harabasz: ↑ lebih baik
if chi_km > chi_h:   votes_km += 1
elif chi_h > chi_km: votes_hier += 1

print('='*52)
print('  PENENTUAN ALGORITMA TERBAIK (VOTING 3 METRIK)')
print('='*52)
print(f'  K-Means        : {votes_km} suara')
print(f'  Hierarchical   : {votes_hier} suara')
print('='*52)

if votes_km > votes_hier:
    best_algo = 'K-Means'
elif votes_hier > votes_km:
    best_algo = 'Hierarchical'
else:
    # Tiebreaker: gunakan Silhouette Score
    best_algo = 'K-Means' if sil_km >= sil_h else 'Hierarchical'
    print(f'  Seri! Tiebreaker Silhouette → {best_algo}')

print(f'\n  ➤  Algoritma Terbaik: {best_algo}  ◄')
print('='*52)

  PENENTUAN ALGORITMA TERBAIK (VOTING 3 METRIK)
  K-Means        : 3 suara
  Hierarchical   : 0 suara

  ➤  Algoritma Terbaik: K-Means  ◄


In [12]:
# ── Cell 14: Simpan Semua Hasil ───────────────────────────────────────────────

# 1. hasil_hierarchical.csv
hier_path = RESULTS_DIR / 'hasil_hierarchical.csv'
df_clean.to_csv(hier_path, index=False)
print(f'Disimpan: {hier_path}  shape={df_clean.shape}')

# 2. metrics_hierarchical.json
mh_path = RESULTS_DIR / 'metrics_hierarchical.json'
with open(mh_path, 'w') as f:
    json.dump(metrics_hier, f, indent=2)
print(f'Disimpan: {mh_path}')

# 3. comparison_table.csv
cmp_path = RESULTS_DIR / 'comparison_table.csv'
df_comparison['best_algorithm'] = best_algo
df_comparison.to_csv(cmp_path, index=False)
print(f'Disimpan: {cmp_path}')

# 4. summary_final.json
summary = {
    'best_algorithm': best_algo,
    'k_optimal':      int(K_OPTIMAL),
    'votes_kmeans':   int(votes_km),
    'votes_hier':     int(votes_hier),
    'kmeans_metrics': metrics_km,
    'hier_metrics':   metrics_hier,
}
summary_path = RESULTS_DIR / 'summary_final.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Disimpan: {summary_path}')

print(f'\n{"="*60}')
print('  RINGKASAN AKHIR')
print(f'{"="*60}')
print(f'  Algoritma Terbaik    : {best_algo}')
print(f'  Jumlah Klaster (k)   : {K_OPTIMAL}')
print(f'  K-Means Silhouette   : {sil_km:.4f}')
print(f'  Hier.   Silhouette   : {sil_h:.4f}')
print(f'  K-Means DBI          : {dbi_km:.4f}')
print(f'  Hier.   DBI          : {dbi_h:.4f}')
print(f'  K-Means CHI          : {chi_km:.2f}')
print(f'  Hier.   CHI          : {chi_h:.2f}')
print(f'{"="*60}')
print('\n✅ Analisis clustering selesai!')

Disimpan: analysis\output\results\hasil_hierarchical.csv  shape=(371, 12)
Disimpan: analysis\output\results\metrics_hierarchical.json
Disimpan: analysis\output\results\comparison_table.csv
Disimpan: analysis\output\results\summary_final.json

  RINGKASAN AKHIR
  Algoritma Terbaik    : K-Means
  Jumlah Klaster (k)   : 3
  K-Means Silhouette   : 0.1816
  Hier.   Silhouette   : 0.1289
  K-Means DBI          : 1.8145
  Hier.   DBI          : 1.9711
  K-Means CHI          : 101.60
  Hier.   CHI          : 85.10

✅ Analisis clustering selesai!


## Interpretasi Profil Klaster Final & Rekomendasi Kebijakan

Interpretasi berikut disusun berdasarkan profil rata-rata 8 fitur proporsi pada Cell 11, dihitung dari `hasil_hierarchical.csv` (k=3, Ward, Silhouette=0,1577, dari `metrics_hierarchical.json`).

---

### Klaster 0 — **Kemiskinan Tinggi** (138 desa, 37,2%)

| Indikator | Nilai Rata-rata |
|-----------|----------------|
| pct_desil1 | 18,14% |
| pct_desil2 | 15,88% |
| pct_desil6_10 | 33,25% |

**Karakteristik:** Desil 1+2 tertinggi (34,02%) dan Desil 6-10 terendah di antara ketiga klaster; proporsi nonaktif (7,24%) juga tertinggi.

**Rekomendasi kebijakan:**
- Program Keluarga Harapan (PKH) diprioritaskan
- Subsidi langsung tunai
- Pemberdayaan ekonomi mikro

---

### Klaster 1 — **Kemiskinan Sedang** (127 desa, 34,2%)

| Indikator | Nilai Rata-rata |
|-----------|----------------|
| pct_desil1 | 16,53% |
| pct_desil2 | 12,49% |
| pct_desil6_10 | 40,13% |

**Karakteristik:** Posisi menengah (Desil 1+2 = 29,02%); Desil 5 tertinggi (16,02%) — kelompok rentan (near-poor).

**Rekomendasi kebijakan:**
- Peningkatan akses layanan dasar
- Program pelatihan vokasional
- Bantuan pangan non-tunai

---

### Klaster 2 — **Kemiskinan Rendah / Sejahtera** (106 desa, 28,6%)

| Indikator | Nilai Rata-rata |
|-----------|----------------|
| pct_desil1 | 8,29% |
| pct_desil2 | 7,49% |
| pct_desil6_10 | 56,47% |

**Karakteristik:** Desil 1+2 terendah (15,78%), Desil 6-10 tertinggi — mayoritas keluarga sejahtera.

**Rekomendasi kebijakan:**
- Fokus pada pencegahan regresif
- Pengembangan UMKM lokal
- Penguatan infrastruktur digital

---

### Wilayah Prioritas Intervensi

> Desa/kelurahan pada **Klaster 0** (kemiskinan tertinggi) terkonsentrasi di kecamatan:
> - Mulak Ulu (13 desa)
> - Pagar Gunung (11 desa)
> - Kikim Selatan (10 desa)
> - Gumay Talang (10 desa)
> - Tanjung Sakti Pumi (9 desa)

merupakan wilayah yang membutuhkan perhatian dan intervensi kebijakan paling mendesak.

---

*Catatan: Hasil clustering ini bersifat deskriptif dan harus dikombinasikan dengan analisis kualitatif lapangan sebelum digunakan sebagai dasar kebijakan.*